In [9]:
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnableParallel, branch
from dataclasses import Field

from  langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel
from typing import Literal, TypedDict, Annotated, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages
from IPython.display import Image,display, Markdown



# This function will load all the variables from the .env file and will
# make them available in the os.environ dictionary (env variables)
load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()
class graph_schema(TypedDict):
    category:str
    messages:Annotated[List,add_messages]

class CategoryOutput(BaseModel):
    category: Literal["billing", "technical", "general"]

system_prompt= "As classifying message expert, u read the message and classify the user intent among 3 category billing/technical/general and set category with right value"

def classify_node(state:graph_schema):
    curr_message=state['messages'][-1]
    curr_check=llm_openai.with_structured_output(CategoryOutput)

    response=curr_check.invoke([SystemMessage(content=system_prompt), curr_message])
    category=response.category
    return {"category":category}

def billing_node(state: graph_schema):
    return {"messages": [AIMessage(content="Billing team will help you")]}
def technical_node(state: graph_schema):
    return {"messages": [AIMessage(content="Tech team will help you")]}

def general_node(state: graph_schema):
    return {"messages": [AIMessage(content="How help you?")]}
def condition(state: graph_schema) -> str:
    category = state['category']

    if category == 'billing':
        return 'billing_node'
    elif category == 'technical':
        return 'technical_node'
    elif category == 'general':
        return 'general_node'
    else:
        raise ValueError("Invalid category")


graph=StateGraph(graph_schema)

graph.add_node("classify_node",classify_node)
graph.add_node("billing_node",billing_node)
graph.add_node("technical_node",technical_node)
graph.add_node("general_node",general_node)

graph.add_edge(START,"classify_node")
graph.add_conditional_edges("classify_node",condition,{"billing_node":"billing_node",
                                                       "technical_node":"technical_node",
                                                       "general_node":"general_node"})
graph.add_edge("billing_node",END)
graph.add_edge("technical_node",END)
graph.add_edge("general_node",END)
f_graph=graph.compile()

# Display Mermaid text instead of trying to render as PNG
print(f_graph.get_graph().draw_mermaid())
result1 = f_graph.invoke({"messages": [HumanMessage(content="My invoice is wrong")]})
result2 = f_graph.invoke({"messages": [HumanMessage(content="My laptop won't start")]})
result3 = f_graph.invoke({"messages": [HumanMessage(content="Hello, how are you?")]})

print("Test 1:", result1['messages'][-1].content)
print("Test 2:", result2['messages'][-1].content)
print("Test 3:", result3['messages'][-1].content)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify_node(classify_node)
	billing_node(billing_node)
	technical_node(technical_node)
	general_node(general_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify_node;
	classify_node -.-> billing_node;
	classify_node -.-> general_node;
	classify_node -.-> technical_node;
	billing_node --> __end__;
	general_node --> __end__;
	technical_node --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CategoryOutput(category='billing'), input_type=CategoryOutput])
  return self.__pydantic_serializer__.to_python(
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CategoryOutput(category='technical'), input_type=CategoryOutput])
  return self.__pydantic_serializer__.to_python(


Test 1: Billing team will help you
Test 2: Tech team will help you
Test 3: How help you?


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CategoryOutput(category='general'), input_type=CategoryOutput])
  return self.__pydantic_serializer__.to_python(


In [14]:
import os
from dotenv import load_dotenv
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal, TypedDict, Annotated, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END
load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()
class RefineState(TypedDict):
    question:   str
    answer:     str
    is_good:    bool
    iterations: int

class GradeOutput(BaseModel):
    is_good: bool

system_prompt= "As question answer evaluator expert, review the answer and tell the answer is good or not and based on it set value is_good with True or False"


def generate_node(state:RefineState):
    question=state.get("question")
    response=llm_openai.invoke(question).content
    return {"answer": response, "iterations": state["iterations"] + 1}

def grade_node(state: RefineState):
    question=state.get("question")
    answer=state.get("answer")
    user_prompt=f"Question asked to user {question}, answer provided by user as {answer}"
    result=llm_openai.with_structured_output(GradeOutput)
    result_grade=result.invoke([SystemMessage(content=system_prompt),HumanMessage(content=user_prompt)])
    return {"is_good": result_grade.is_good}

def condition(state: RefineState) -> str:
    category = state['is_good']
    interation= state['iterations']
    if category == True or interation>=3:
        return 'end'
    else:
        return "generate_node"


graph=StateGraph(RefineState)

graph.add_node("generate_node",generate_node)
graph.add_node("grade_node",grade_node)

graph.add_edge(START,"generate_node")
graph.add_edge("generate_node","grade_node")
graph.add_conditional_edges("grade_node",condition,{"generate_node":"generate_node",                       "end":END})
f_graph=graph.compile()
result = f_graph.invoke({
    "question": "What is a LangGraph reducer in one sentence?",
    "answer": "",
    "is_good": False,
    "iterations": 0
})
print(f"Final answer: {result['answer']}")
print(f"Iterations taken: {result['iterations']}")
print(f"Passed quality check: {result['is_good']}")

Final answer: A LangGraph reducer is a function that aggregates the outputs of multiple graph nodes into a single combined result during the graph’s execution.
Iterations taken: 1
Passed quality check: True


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GradeOutput(is_good=True), input_type=GradeOutput])
  return self.__pydantic_serializer__.to_python(


In [18]:
import operator
import os
from dotenv import load_dotenv
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal, TypedDict, Annotated, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END
load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()
class ResearchState(TypedDict):
    topic:    str
    findings: Annotated[List[str], operator.add]  # reducer merges both nodes
    summary:  str

class FindingsList(BaseModel):
    items: List[str]

system_pros_prompt= "As researcher you find the top 2 pros based on selected topic"
system_cons_prompt= "As researcher you find the top 2 cons based on selected topic"

system_summary_prompt= "As researcher read all finding pros and cons based on selected topic and writes a balanced summary"



def pro_node(state:ResearchState):
    question=state.get("topic")
    pros_finder=llm_openai.with_structured_output(FindingsList)
    response=pros_finder.invoke([SystemMessage(content=system_pros_prompt), HumanMessage(content=question)])
    return {"findings": response.items}

def con_node(state:ResearchState):
    question=state.get("topic")
    pros_finder=llm_openai.with_structured_output(FindingsList)
    response=pros_finder.invoke([SystemMessage(content=system_cons_prompt), HumanMessage(content=question)])
    return {"findings": response.items}

def synthesize_node(state: ResearchState):
    answer=state.get("findings")
    response=llm_openai.invoke([SystemMessage(content=system_summary_prompt), HumanMessage(content='\n'.join(answer))]).content
    return {"summary": response}




graph=StateGraph(ResearchState)
graph.add_node("pro_node", pro_node)
graph.add_node("synthesize_node", synthesize_node)
graph.add_node("con_node", con_node)

graph.add_edge(START, "pro_node")
graph.add_edge(START, "con_node")
graph.add_edge("pro_node", "synthesize_node")
graph.add_edge("con_node", "synthesize_node")
graph.add_edge("synthesize_node", END)
f_graph=graph.compile()
result = f_graph.invoke({"topic": "Working from home", "findings": [], "summary": ""})
print(f"All findings ({len(result['findings'])} total):")
for f in result['findings']:
    print(f"  - {f}")
print(f"\nSummary: {result['summary']}")

C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=FindingsList(items=['Flex...g overall efficiency.']), input_type=FindingsList])
  return self.__pydantic_serializer__.to_python(
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=FindingsList(items=['1. *...aining personal time.']), input_type=FindingsList])
  return self.__pydantic_serializer__.to_python(


All findings (4 total):
  - 1. **Isolation and reduced collaboration** – Remote workers miss out on spontaneous face‑to‑face interactions, making it harder to build relationships, share ideas quickly, and stay aligned with team culture.
  - 2. **Blurring of work‑life boundaries** – Without a clear physical separation, many people find it difficult to “switch off,” leading to longer hours, burnout, and challenges in maintaining personal time.
  - Flexibility & Work‑Life Balance – Remote work lets employees set their own schedules, reduce commute time, and more easily integrate personal responsibilities (family, health, hobbies) with professional duties, leading to higher satisfaction and lower stress.
  - Cost & Time Savings – Eliminating daily travel cuts transportation expenses, fuel, and parking fees while freeing up hours that can be redirected to productive work, personal development, or leisure, ultimately boosting overall efficiency.

Summary: **Balanced Summary of Remote‑Work Be